In [1]:
import requests
from dateutil import parser
from datetime import datetime
import pytz
import re
from collections import defaultdict

url = "https://gql.novig.us/v1/graphql"

query = """
query {
  event(where: {_and: [
      {_or: [
        {status: {_eq: "OPEN_PREGAME"}},
        {status: {_eq: "OPEN_INGAME"}}
      ]},
      {game: {league: {_eq: "MLB"}}}
    ]}) {
    id
    description
    game {
      scheduled_start
    }
    markets {
      description
      updated_at
      is_consensus
      outcomes(where: {_or: [{last: {_is_null: false}}, {available: {_is_null: false}}]}) {
        description
        available
        last
        updated_at
      }
    }
  }
}
"""

def price_to_american(price):
    try:
        if price is None:
            return "N/A"
        price = float(price)
        if price == 0.5:
            return "+100"
        elif price < 0.5:
            odds = (1 / price - 1) * 100
            return f"+{int(round(odds))}"
        else:
            odds = -100 / (1 / price - 1)
            return f"{int(round(odds))}"
    except Exception:
        return "N/A"

# Helper to determine if the favorite is at home
def is_fav_home(fav_abbr, home_team_full):
    return fav_abbr.upper() in home_team_full.upper()

response = requests.post(url, json={'query': query})
data = response.json()

eastern = pytz.timezone('US/Eastern')
now_eastern = datetime.now(eastern)
today_est = now_eastern.date()
now_est = now_eastern

results = []
spread_regex = re.compile(r'([A-Z]{2,3}) ([+-]?\d+\.5)')

for event in data['data']['event']:
    desc = event['description']
    # Get teams
    if " @ " in desc:
        away_team, home_team = desc.split(" @ ")
    else:
        away_team, home_team = "", ""
    game_time_utc = parser.parse(event['game']['scheduled_start'])
    game_time_est = game_time_utc.astimezone(eastern)
    if game_time_est.date() != today_est or game_time_est < now_est:
        continue
    for market in event['markets']:
        if not market.get("is_consensus"):
            continue
        for outcome in market['outcomes']:
            match = spread_regex.match(outcome['description'])
            if not match:
                continue
            team = match.group(1)
            line = float(match.group(2))
            market_timestamp = outcome.get('updated_at') or market.get('updated_at') or "N/A"
            try:
                ts = parser.parse(market_timestamp)
            except Exception:
                ts = None
            results.append({
                "teams": desc,
                "away_team": away_team,
                "home_team": home_team,
                "team": team,
                "line": line,
                "spread_line": outcome['description'],
                "spread_price": outcome['available'],
                "game_time_est": game_time_est.isoformat(),
                "market_timestamp": market_timestamp,
                "timestamp_dt": ts,
                "market_desc": market['description']
            })

# Group by matchup and absolute line
output = defaultdict(dict)
for r in results:
    key = (r['teams'], abs(r['line']))
    output[key][r['line']] = r

for (teams, abs_line), spread_pair in output.items():
    # We want both the favorite (-line) and underdog (+line)
    fav = spread_pair.get(-abs_line)
    dog = spread_pair.get(abs_line)
    if not fav or not dog:
        continue  # Only output if both sides exist

    fav_team = fav['team']
    fav_at_home = 1 if is_fav_home(fav_team, fav['home_team']) else 0

    print(
        f"{fav_team} (fav) vs {dog['team']} (dog) | HomeFav: {fav_at_home} | "
        f"Line: {fav['line']}, FavPrice: {fav['spread_price']} ({price_to_american(fav['spread_price'])}), "
        f"DogLine: {dog['line']}, DogPrice: {dog['spread_price']} ({price_to_american(dog['spread_price'])}) | "
        f"Game Time EST: {fav['game_time_est']} | Market: {fav['market_desc']:<15} | Market Timestamp: {fav['market_timestamp']}"
    )


CHC (fav) vs COL (dog) | HomeFav: 0 | Line: -1.5, FavPrice: 0.568 (-131), DogLine: 1.5, DogPrice: 0.467 (+114) | Game Time EST: 2025-05-27T20:05:00-04:00 | Market: CHC -1.5        | Market Timestamp: 2025-05-27T23:51:25.396+00:00
NYY (fav) vs LAA (dog) | HomeFav: 0 | Line: -1.5, FavPrice: 0.56 (-127), DogLine: 1.5, DogPrice: 0.455 (+120) | Game Time EST: 2025-05-27T21:38:00-04:00 | Market: LAA +1.5        | Market Timestamp: 2025-05-27T23:46:22.267+00:00
SD (fav) vs MIA (dog) | HomeFav: 0 | Line: -1.5, FavPrice: 0.419 (+139), DogLine: 1.5, DogPrice: 0.595 (-147) | Game Time EST: 2025-05-27T21:40:00-04:00 | Market: SD -1.5         | Market Timestamp: 2025-05-27T23:51:36.377+00:00
HOU (fav) vs OAK (dog) | HomeFav: 1 | Line: -1.5, FavPrice: 0.51 (-104), DogLine: 1.5, DogPrice: 0.512 (-105) | Game Time EST: 2025-05-27T20:10:00-04:00 | Market: HOU -1.5        | Market Timestamp: 2025-05-27T23:47:49.789+00:00
ARI (fav) vs PIT (dog) | HomeFav: 1 | Line: -1.5, FavPrice: 0.555 (-125), DogLine: 

In [2]:
import os
import requests
import pandas as pd
from dateutil import parser
from datetime import datetime
import pytz
import re
from collections import defaultdict

# === CONFIG ===
url = "https://gql.novig.us/v1/graphql"
folder = "novig-odds"
os.makedirs(folder, exist_ok=True)
today_str = datetime.now().strftime("%Y-%m-%d")
filename = f"novig_spreads_{today_str}.csv"
output_path = os.path.join(folder, filename)

# === GRAPHQL QUERY ===
query = """
query {
  event(where: {_and: [
      {_or: [
        {status: {_eq: "OPEN_PREGAME"}},
        {status: {_eq: "OPEN_INGAME"}}
      ]},
      {game: {league: {_eq: "MLB"}}}
    ]}) {
    id
    description
    game {
      scheduled_start
    }
    markets {
      description
      updated_at
      is_consensus
      outcomes(where: {_or: [{last: {_is_null: false}}, {available: {_is_null: false}}]}) {
        description
        available
        last
        updated_at
      }
    }
  }
}
"""

def price_to_american(price):
    try:
        if price is None:
            return "N/A"
        price = float(price)
        if price == 0.5:
            return "+100"
        elif price < 0.5:
            odds = (1 / price - 1) * 100
            return f"+{int(round(odds))}"
        else:
            odds = -100 / (1 / price - 1)
            return f"{int(round(odds))}"
    except Exception:
        return "N/A"

def is_fav_home(fav_abbr, home_team_full):
    return fav_abbr.upper() in home_team_full.upper()

# === API CALL ===
response = requests.post(url, json={'query': query})
data = response.json()

eastern = pytz.timezone('US/Eastern')
now_eastern = datetime.now(eastern)
today_est = now_eastern.date()
now_est = now_eastern

results = []
spread_regex = re.compile(r'([A-Z]{2,3}) ([+-]?\d+\.5)')

for event in data['data']['event']:
    desc = event['description']
    if " @ " not in desc:
        continue
    away_team, home_team = desc.split(" @ ")
    game_time_utc = parser.parse(event['game']['scheduled_start'])
    game_time_est = game_time_utc.astimezone(eastern)
    if game_time_est.date() != today_est or game_time_est < now_est:
        continue
    for market in event['markets']:
        if not market.get("is_consensus"):
            continue
        for outcome in market['outcomes']:
            match = spread_regex.match(outcome['description'])
            if not match:
                continue
            team = match.group(1)
            line = float(match.group(2))
            market_timestamp = outcome.get('updated_at') or market.get('updated_at') or "N/A"
            try:
                ts = parser.parse(market_timestamp)
            except Exception:
                ts = None
            results.append({
                "teams": desc,
                "away_team": away_team,
                "home_team": home_team,
                "team": team,
                "line": line,
                "spread_line": outcome['description'],
                "spread_price": outcome['available'],
                "game_time_est": game_time_est.isoformat(),
                "market_timestamp": market_timestamp,
                "timestamp_dt": ts,
                "market_desc": market['description']
            })

# === PROCESSING FAVORITES/UNDERDOGS ===
rows = []
output = defaultdict(dict)
for r in results:
    key = (r['teams'], abs(r['line']))
    output[key][r['line']] = r

for (teams, abs_line), spread_pair in output.items():
    fav = spread_pair.get(-abs_line)
    dog = spread_pair.get(abs_line)
    if not fav or not dog:
        continue

    fav_team = fav['team']
    fav_at_home = 1 if is_fav_home(fav_team, fav['home_team']) else 0

    rows.append({
        "Date": today_str,
        "Favorite": fav_team,
        "Underdog": dog['team'],
        "Fav Price (Decimal)": fav['spread_price'],
        "Dog Price (Decimal)": dog['spread_price'],
        "Fav Price (American)": price_to_american(fav['spread_price']),
        "Dog Price (American)": price_to_american(dog['spread_price']),
        "Line": fav['line'],
        "Fav at Home?": fav_at_home,
        "Game Time (EST)": fav['game_time_est'],
        "Market": fav['market_desc'],
        "Market Timestamp": fav['market_timestamp']
    })

# === SAVE TO CSV ===
df = pd.DataFrame(rows)
df.to_csv(output_path, index=False)
print(f"✅ Saved {len(df)} spreads to {output_path}")
print(df.head())


✅ Saved 7 spreads to novig-odds/novig_spreads_2025-05-27.csv
         Date Favorite Underdog  Fav Price (Decimal)  Dog Price (Decimal)  \
0  2025-05-27      CHC      COL                0.568                0.467   
1  2025-05-27      NYY      LAA                0.560                0.455   
2  2025-05-27       SD      MIA                0.419                0.595   
3  2025-05-27      HOU      OAK                0.510                0.512   
4  2025-05-27      ARI      PIT                0.555                0.474   

  Fav Price (American) Dog Price (American)  Line  Fav at Home?  \
0                 -131                 +114  -1.5             0   
1                 -127                 +120  -1.5             0   
2                 +139                 -147  -1.5             0   
3                 -104                 -105  -1.5             1   
4                 -125                 +111  -1.5             1   

             Game Time (EST)    Market               Market Timestamp  
0